# 1. Data Audit: `orders` Table

An inital data profiling was performed on the olist_orders_dataset.csv file using Pandas to define the architecture for the PostgreSQL database.

## Technical Findings:
* **Data Volume:** 99,441 records
* **Column Integrity:** Identify **8 columns** with 3 columns (time) of null values.

### Cardinality  Analysis:
* **`order_id`:** Contains 99,441 uniques values, confirming its suitability as the **Primary Key (PK)**.
* **`customer_id`:** Contains 99,441 uniques values with zero null. It is the ideal candidate for a Foreign Key (FK) to link with the customers table.

In [6]:
import pandas as pd

df_orders = pd.read_csv('../data/olist_orders_dataset.csv')

data_quality = pd.DataFrame({
    'Type': df_orders.dtypes,
    'Non Nulls': df_orders.count(),
    'Nulls': df_orders.isnull().sum(),
    'Unique Values': df_orders.nunique(),
    'Duplicated': len(df_orders) - df_orders.nunique()
})

data_quality

,Type,Non Nulls,Nulls,Unique Values,Duplicated
order_id,object,99441,0,99441,0
customer_id,object,99441,0,99441,0
order_status,object,99441,0,8,99433
order_purchase_timestamp,object,99441,0,98875,566
order_approved_at,object,99281,160,90733,8708
order_delivered_carrier_date,object,97658,1783,81018,18423
order_delivered_customer_date,object,96476,2965,95664,3777
order_estimated_delivery_date,object,99441,0,459,98982


# 2. Data Quality Assessment: Schema Validation

### Issue: Incorrect Data Types for Temporal Columns
During the initial inspection, it was identified that all timestamp-related columns are currently stored as `object` (strings).

**Impact:** * Numerical operations (calculating delivery lags, SLAs, or durations) are not possible.
* Increased memory consumption.
* Risk of logical inconsistencies in time-series analysis.

**Action Plan:**
Convert the following columns to `datetime64[ns]` before proceeding to the Transformation (ETL) phase:
* `order_purchase_timestamp`
* `order_approved_at`
* `order_delivered_carrier_date`
* `order_delivered_customer_date`
* `order_estimated_delivery_date`

In [7]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

print(df_orders[date_columns].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


## 1.3. Integrity of column "Order_status"

An inital cosistency analysis was performed between logistic of purchase status and the customer delivery date.

* **Logical consistency in intermediate states:** The states `aprobed`, `created`, `shipped`, `processing`, `invoiced` and `unavailable` show a 100% null rate in the delivery date column, this is expected and correct. as these orders have not yet completed the full delivery cycle.

* **Delivered Orders Anomaly:** Within the `delivered` status **8 records** were identified as "delivered" but lack a completion timestamp (NaN). This represents a data integrity error that should be flagged for exclusion or further investigation.
* **Post-Delivery Cancellations:** There are **6 records** in `canceled` status that actually possess a delivery timestamp This suggests Edge Cases where cancellations were processed administratively after the product had already reached the customer.

In [5]:
##############################################################################################################

# GROUP BY STATUS AND SHOW COLUMNS WHITOUT A DATE AND COLUMNS WITH A CUSTOMER DALIVERY DATE

##############################################################################################################

data_status = df_orders.groupby('order_status')['order_delivered_customer_date'].agg(
    Orders_Total = 'size',
    Undelivered = lambda x: x.isnull().sum(),
    Delivered = 'count'
)

data_status


,Orders_Total,Undelivered,Delivered
order_status,,,
approved,2,2,0
canceled,625,619,6
created,5,5,0
delivered,96478,8,96470
invoiced,314,314,0
processing,301,301,0
shipped,1107,1107,0
unavailable,609,609,0
